# 02 Data Preprocessing Walkthrough

This notebook demonstrates the preprocessing pipeline: normalising chest X-rays to [-1024, 1024] range (torchxrayvision expectations) and cleaning report text.

In [ ]:
import os
import sys
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Add project root to path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(''))))
from src.preprocessing import normalize_xray, clean_report_text, extract_findings_section, extract_impression_section
from src.data_loader import IUXRayDataset, extract_labels_from_report
from src.config import get_config, PATHOLOGY_LABELS

## 1. Image Preprocessing Demo

In [ ]:
# Load a mock raw image
raw_img_arr = np.random.randint(0, 255, (256, 256), dtype=np.uint8)
raw_image = Image.fromarray(raw_img_arr)

# Preprocess to model-ready tensor
tensor = normalize_xray(raw_image)
print(f"Raw image dimensions: {raw_image.size}")
print(f"Preprocessed tensor shape: {tensor.shape}")
print(f"Tensor scale: min={tensor.min().item():.2f}, max={tensor.max().item():.2f} (expects [-1024, 1024])")

## 2. Text Preprocessing & Section Extraction

In [ ]:
raw_report = """
XML REPORT: STUDY DATE: 2026-06-10
FINDINGS:
Cardiomegaly is identified with cardiothoracic ratio enlarged. Lungs are clear without focal infiltrates.

IMPRESSION:
Enlarged cardiac silhouette.
"""

cleaned = clean_report_text(raw_report)
findings = extract_findings_section(raw_report)
impression = extract_impression_section(raw_report)

print("Cleaned Full Text:")
print(cleaned)
print("\nExtracted Findings:")
print(findings)
print("\nExtracted Impression:")
print(impression)

## 3. Label Extraction from Text

In [ ]:
labels = extract_labels_from_report(raw_report, PATHOLOGY_LABELS)
print("Extracted Labels:")
for label, val in zip(PATHOLOGY_LABELS, labels):
    if val != 0.0:
        print(f"  {label}: {val} (1.0 = positive, -1.0 = uncertain)")